In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
def install_dependencies():
    import os

    # 1. Kill Hugging Face progress bars
    os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
    os.environ["TRANSFORMERS_VERBOSITY"] = "error"

    # 2. Fix the transformers crash by removing the broken optional dependency
    os.system('pip uninstall -y torchao --quiet')

    # 3. Install required dependencies
    os.system('pip install av dlib imageio yacs==0.1.8 gradio opencv-python librosa transformers accelerate --quiet')
    os.system('pip uninstall -y scipy --quiet')
    os.system('pip install scipy==1.11.4 --quiet')

    # 4. Patch DreamTalk to use imageio instead of torchvision.io
    utils_path = "/content/drive/MyDrive/dreamtalk_main/generators/utils.py"
    if os.path.exists(utils_path):
        with open(utils_path, "r") as f:
            code = f.read()

        target_line = "torchvision.io.write_video(silent_video_path, transformed_imgs.cpu(), fps)"
        new_line = "import imageio; imageio.mimsave(silent_video_path, transformed_imgs.cpu().numpy(), fps=fps)"

        if target_line in code:
            code = code.replace(target_line, new_line)
            with open(utils_path, "w") as f:
                f.write(code)
            print("✅ Patched DreamTalk utils.py to fix torchvision error!")

    print("Step 1 done")

install_dependencies()

✅ Patched DreamTalk utils.py to fix torchvision error!
Step 1 done


In [ ]:

# --- 2. Pipeline Functions ---

# Create required directories
def create_directories():
    import os
    base_path = "/content/drive/MyDrive"
    dreamtalk_path = os.path.join(base_path, "dreamtalk_main")

    # Create all required directories
    directories = [
        os.path.join(dreamtalk_path, "data", "src_img", "uncropped"),
        os.path.join(dreamtalk_path, "data", "audio"),
        os.path.join(dreamtalk_path, "output_video"),
        os.path.join(dreamtalk_path, "output_video", "enhanced_frames"),
        os.path.join(dreamtalk_path, "output_video", "codeformer_frames"),
        os.path.join(dreamtalk_path, "output_video", "enhanced_frames", "final_results")
    ]

    for directory in directories:
        os.makedirs(directory, exist_ok=True)
        print(f"✅ Created directory: {directory}")

# Create directories first
create_directories()

def setup_paths(job_id, image_path, audio_path, output_video_path):
    import os
    base_path = "/content/drive/MyDrive"
    dreamtalk_path = os.path.join(base_path, "dreamtalk_main")
    real_esrgan_path = os.path.join(base_path, "Real-ESRGAN-master")
    codeformer_path = os.path.join(base_path, "CodeFormer-master")
    output_name = job_id
    enhanced_base = os.path.join(dreamtalk_path, "output_video", "enhanced_frames")
    output_dir = os.path.join(dreamtalk_path, "output_video", "codeformer_frames")
    enhanced_dir = os.path.join(enhanced_base, "final_results")
    final_output = os.path.join(dreamtalk_path, "output_video", f"{output_name}_enhanced.mp4")
    return {
        'base_path': base_path,
        'dreamtalk_path': dreamtalk_path,
        'real_esrgan_path': real_esrgan_path,
        'codeformer_path': codeformer_path,
        'output_name': output_name,
        'image_path': image_path,
        'audio_path': audio_path,
        'output_video_path': output_video_path,
        'enhanced_base': enhanced_base,
        'output_dir': output_dir,
        'enhanced_dir': enhanced_dir,
        'final_output': final_output
    }

def generate_talking_head_video(paths):
    import os
    os.chdir(paths['dreamtalk_path'])

    # Check if inference_for_demo_video is a folder or file
    inference_path = os.path.join(paths['dreamtalk_path'], "inference_for_demo_video")
    if os.path.isdir(inference_path):
        # It's a folder, look for the main script
        script_path = os.path.join(inference_path, "inference_for_demo_video.py")
        if not os.path.exists(script_path):
            # Try alternative names
            for file in os.listdir(inference_path):
                if file.endswith('.py') and 'inference' in file.lower():
                    script_path = os.path.join(inference_path, file)
                    break
    else:
        # It's a file
        script_path = inference_path + ".py"

    # Find style clip and pose files
    style_clip_path = os.path.join(paths['dreamtalk_path'], 'data', 'style_clip', '3DMM', 'W009_front_neutral_level1_001.mat')
    pose_path = os.path.join(paths['dreamtalk_path'], 'data', 'pose', 'RichardShelby_front_neutral_level1_001.mat')

    # If default files don't exist, try to find alternatives
    if not os.path.exists(style_clip_path):
        style_clip_dir = os.path.join(paths['dreamtalk_path'], 'data', 'style_clip')
        if os.path.exists(style_clip_dir):
            for file in os.listdir(style_clip_dir):
                if file.endswith('.mat'):
                    style_clip_path = os.path.join(style_clip_dir, file)
                    break

    if not os.path.exists(pose_path):
        pose_dir = os.path.join(paths['dreamtalk_path'], 'data', 'pose')
        if os.path.exists(pose_dir):
            for file in os.listdir(pose_dir):
                if file.endswith('.mat'):
                    pose_path = os.path.join(pose_dir, file)
                    break

    print(f"Using script: {script_path}")
    print(f"Using style clip: {style_clip_path}")
    print(f"Using pose: {pose_path}")

    # os.system(f"""
    # python {script_path} \
    #   --wav_path \"{paths['audio_path']}\" \
    #   --style_clip_path \"{style_clip_path}\" \
    #   --pose_path \"{pose_path}\" \
    #   --image_path \"{paths['image_path']}\" \
    #   --cfg_scale 1.0 \
    #   --max_gen_len 40 \
    #   --output_name \"{paths['output_name']}\""""
    # )
    import subprocess
    cmd = [
        "python", script_path,
        "--wav_path", paths['audio_path'],
        "--style_clip_path", style_clip_path,
        "--pose_path", pose_path,
        "--image_path", paths['image_path'],
        "--cfg_scale", "1.0",
        "--max_gen_len", "40",
        "--output_name", paths['output_name'],
        "--device", "cpu"

    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print("DreamTalk STDOUT:\n", result.stdout)
    print("DreamTalk STDERR:\n", result.stderr)
    print("Step 3 done")

def install_real_esrgan(paths):
    import os
    os.chdir(paths['real_esrgan_path'])
    os.system('pip install -r requirements.txt')
    os.system('python setup.py develop')
    print("Step 4 done")

def install_codeformer(paths):
    import os
    os.chdir(paths['codeformer_path'])
    os.system('pip install -r requirements.txt')
    os.system('python basicsr/setup.py develop')
    os.system('pip install facexlib gradio insightface==0.7.3')
    os.system('pip install xformers==0.0.22 triton==2.0.0')
    os.system('pip install moviepy==1.')
    # REMOVED THE PYTORCH DOWNGRADE LINE FROM HERE
    print("Step 5 done")

def extract_frames(video_path, output_dir):
    import os
    import cv2
    import glob
    if os.path.exists(output_dir):
        frame_files = glob.glob(os.path.join(output_dir, "frame_*.png"))
        for file in frame_files:
            os.remove(file)
    else:
        os.makedirs(output_dir, exist_ok=True)
    vidcap = cv2.VideoCapture(video_path)
    success, image = vidcap.read()
    count = 0
    while success:
        cv2.imwrite(f"{output_dir}/frame_{count:05d}.png", image)
        success, image = vidcap.read()
        count += 1
    vidcap.release()
    print("Step 6 done")

def enhance_frames_with_codeformer(paths):
    import os
    import glob
    final_results_dir = os.path.join(paths['enhanced_base'], "final_results")
    if os.path.exists(final_results_dir):
        enhanced_files = glob.glob(os.path.join(final_results_dir, "frame_*.png"))
        for file in enhanced_files:
            os.remove(file)
    os.chdir(paths['codeformer_path'])
    os.system(f"""
    python inference_codeformer.py \
      -i \"{paths['output_dir']}\" \
      -o \"{paths['enhanced_base']}\" \
      -w 0.7 \
      --face_upsample \
      --bg_upsampler realesrgan""")
    print("Step 7 done")

def frames_to_video_with_audio(frames_dir, original_video, output_video):
    import cv2
    import glob
    import os
    from moviepy.editor import VideoFileClip, AudioFileClip
    from tqdm import tqdm
    if os.path.exists(output_video):
        os.remove(output_video)
    frame_files = sorted(glob.glob(f"{frames_dir}/frame_*.png"))
    frame = cv2.imread(frame_files[0])
    height, width, _ = frame.shape
    vidcap = cv2.VideoCapture(original_video)
    fps = vidcap.get(cv2.CAP_PROP_FPS)
    vidcap.release()
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    temp_video = "temp_video.mp4"
    out = cv2.VideoWriter(temp_video, fourcc, fps, (width, height))
    for file in tqdm(frame_files):
        img = cv2.imread(file)
        out.write(img)
    out.release()
    video = VideoFileClip(temp_video)
    audio = AudioFileClip(original_video)
    video.set_audio(audio).write_videofile(output_video, codec='libx264')
    os.remove(temp_video)
    print("Step 8 done")

def run_pipeline(job_id, image_path, audio_path, output_video_path):
    paths = setup_paths(job_id, image_path, audio_path, output_video_path)
    generate_talking_head_video(paths)
    if not os.path.exists(paths['output_video_path']):
        print(f"Error: Talking head video not created at {paths['output_video_path']}")
        return
    install_real_esrgan(paths)
    install_codeformer(paths)
    extract_frames(paths['output_video_path'], paths['output_dir'])
    if not os.listdir(paths['output_dir']):
        print(f"Error: No frames extracted to {paths['output_dir']}")
        return
    enhance_frames_with_codeformer(paths)
    if not os.listdir(paths['enhanced_dir']):
        print(f"Error: No enhanced frames found in {paths['enhanced_dir']}")
        return
    frames_to_video_with_audio(
        frames_dir=paths['enhanced_dir'],
        original_video=paths['output_video_path'],
        output_video=paths['final_output']
    )
    if not os.path.exists(paths['final_output']):
        print(f"Error: Final video not created at {paths['final_output']}")
        return
    print(f"✔ Final video created successfully for job {job_id}: {paths['final_output']}")

# --- 3. Polling Loop ---
import os
import time
SRC_IMG_DIR = "/content/drive/MyDrive/dreamtalk_main/data/src_img/uncropped"
AUDIO_DIR = "/content/drive/MyDrive/dreamtalk_main/data/audio"
OUTPUT_VIDEO_DIR = "/content/drive/MyDrive/dreamtalk_main/output_video"
def get_job_ids():
    try:
        # Check if directories exist
        if not os.path.exists(SRC_IMG_DIR):
            print(f"❌ Image directory not found: {SRC_IMG_DIR}")
            return []
        if not os.path.exists(AUDIO_DIR):
            print(f"❌ Audio directory not found: {AUDIO_DIR}")
            return []
        if not os.path.exists(OUTPUT_VIDEO_DIR):
            print(f"❌ Output directory not found: {OUTPUT_VIDEO_DIR}")
            return []

        # Get files from each directory
        img_files = {os.path.splitext(f)[0] for f in os.listdir(SRC_IMG_DIR) if f.endswith('.jpg')}
        audio_files = {os.path.splitext(f)[0] for f in os.listdir(AUDIO_DIR) if f.endswith('.wav')}
        done_files = {os.path.splitext(f)[0] for f in os.listdir(OUTPUT_VIDEO_DIR) if f.endswith('_enhanced.mp4')}

        # Find matching job IDs
        matching_jobs = list((img_files & audio_files) - done_files)

        if matching_jobs:
            print(f"🎯 Found {len(matching_jobs)} jobs to process: {matching_jobs}")

        return matching_jobs

    except Exception as e:
        print(f"❌ Error in get_job_ids: {e}")
        return []
while True:
    job_ids = get_job_ids()
    if not job_ids:
        print("No new jobs found. Waiting...")
        time.sleep(10)
        continue
    for job_id in job_ids:
        print(f"Processing job: {job_id}")
        image_path = os.path.join(SRC_IMG_DIR, f"{job_id}.jpg")
        audio_path = os.path.join(AUDIO_DIR, f"{job_id}.wav")
        output_video_path = os.path.join(OUTPUT_VIDEO_DIR, f"{job_id}.mp4")
        run_pipeline(job_id, image_path, audio_path, output_video_path)
    print("Waiting for new jobs...")
    time.sleep(10)

✅ Created directory: /content/drive/MyDrive/dreamtalk_main/data/src_img/uncropped
✅ Created directory: /content/drive/MyDrive/dreamtalk_main/data/audio
✅ Created directory: /content/drive/MyDrive/dreamtalk_main/output_video
✅ Created directory: /content/drive/MyDrive/dreamtalk_main/output_video/enhanced_frames
✅ Created directory: /content/drive/MyDrive/dreamtalk_main/output_video/codeformer_frames
✅ Created directory: /content/drive/MyDrive/dreamtalk_main/output_video/enhanced_frames/final_results
🎯 Found 1 jobs to process: ['22cd0608-618d-4179-9c76-c8a3a1accc47']
Processing job: 22cd0608-618d-4179-9c76-c8a3a1accc47
Using script: /content/drive/MyDrive/dreamtalk_main/inference_for_demo_video.py
Using style clip: /content/drive/MyDrive/dreamtalk_main/data/style_clip/3DMM/W009_front_neutral_level1_001.mat
Using pose: /content/drive/MyDrive/dreamtalk_main/data/pose/RichardShelby_front_neutral_level1_001.mat
DreamTalk STDOUT:
 
DreamTalk STDERR:
 ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copy

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':

100%|██████████| 120/120 [00:05<00:00, 21.19it/s]


Moviepy - Building video /content/drive/MyDrive/dreamtalk_main/output_video/22cd0608-618d-4179-9c76-c8a3a1accc47_enhanced.mp4.
MoviePy - Writing audio in 22cd0608-618d-4179-9c76-c8a3a1accc47_enhancedTEMP_MPY_wvf_snd.mp3


MoviePy - Done.
Moviepy - Writing video /content/drive/MyDrive/dreamtalk_main/output_video/22cd0608-618d-4179-9c76-c8a3a1accc47_enhanced.mp4



Moviepy - Done !
Moviepy - video ready /content/drive/MyDrive/dreamtalk_main/output_video/22cd0608-618d-4179-9c76-c8a3a1accc47_enhanced.mp4
Step 8 done
✔ Final video created successfully for job 22cd0608-618d-4179-9c76-c8a3a1accc47: /content/drive/MyDrive/dreamtalk_main/output_video/22cd0608-618d-4179-9c76-c8a3a1accc47_enhanced.mp4
Waiting for new jobs...
🎯 Found 1 jobs to process: ['22cd0608-618d-4179-9c76-c8a3a1accc47']
Processing job: 22cd0608-618d-4179-9c76-c8a3a1accc47
Using script: /content/drive/MyDrive/dreamtalk_main/inference_for_demo_video.py
Using style clip: /content/drive/MyDrive/dreamtalk_main/data/style_clip/3DMM/W009_front_neutral_level1_001.mat
Using pose: /content/drive/MyDrive/dreamtalk_main/data/pose/RichardShelby_front_neutral_level1_001.mat


KeyboardInterrupt: 

In [ ]:
from google.colab import drive
drive.flush_and_unmount()

Drive not mounted, so nothing to flush and unmount.


In [ ]:
# import os
# base_path = "/content/drive/MyDrive"
# dreamtalk_path = os.path.join(base_path, "dreamtalk_main")
# %cd {dreamtalk_path}
# !python inference_for_demo_video.py \
# --wav_path data/audio/sad_10.wav \
# --style_clip_path data/style_clip/3DMM/W009_front_sad_level3_001.mat \
# --pose_path data/pose/RichardShelby_front_neutral_level1_001.mat \
# --image_path data/src_img/uncropped/anushka.jpg \
# --cfg_scale 1.0 \
# --max_gen_len 40 \
# --output_name "anush"

In [ ]:
!rm -rf /content/drive

In [ ]:
import os
print("Images:", os.listdir("/content/drive/MyDrive/dreamtalk_main/data/src_img/uncropped"))
print("Audio:", os.listdir("/content/drive/MyDrive/dreamtalk_main/data/audio"))